# Système de recommandation produit-à-produit pour le e-commerce

Comparaison de trois approches de recommandation basée sur le contenu, sur un catalogue de 50 000 produits vestimentaires ne contenant aucune interaction utilisateur (pas de clics, pas d'achats, pas de notes) :

1. **KNN hybride** — similarité de contenu (One-Hot + cosinus) combinée à une similarité de prix
2. **Autoencoder multi-tâches** — représentation apprise par reconstruction (têtes softmax par variable catégorielle + tête numérique)
3. **Neural Embedding (Deep Metric Learning)** — embeddings appris directement par Triplet Loss, avec négatifs difficiles

Chaque approche est évaluée sur les mêmes métriques : cohérence de catégorie, cohérence de sous-catégorie, écart relatif de prix.


## 1. Import des librairies

In [ ]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, callbacks

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity


## 2. Chargement des données

Adapter `DATA_PATH` à l'environnement d'exécution (Kaggle, Colab, local...).


In [ ]:
DATA_PATH = "women_clothing_50k.csv"  # <-- adapter ce chemin selon l'environnement

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
df.info()


## 3. Approche 1 — KNN hybride

Le score combine une similarité de contenu (distance cosinus sur les caractéristiques catégorielles encodées en One-Hot) et une similarité de prix relative au prix du produit cible :

$$\text{FinalScore} = 0.75 \cdot \text{ContentSimilarity} + 0.25 \cdot \text{PriceSimilarity}$$


In [ ]:
knn_features = ['category', 'subcategory', 'material', 'color', 'season', 'style']

encoder = OneHotEncoder(handle_unknown='ignore')
knn_feature_matrix = encoder.fit_transform(df[knn_features])

knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10)
knn_model.fit(knn_feature_matrix)

print(knn_feature_matrix.shape)


In [ ]:
def recommend_hybrid(product_id, n_recommendations=5):
    """KNN hybride : similarité de contenu (cosinus, One-Hot) + similarité de prix."""
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]
    target = df.iloc[product_index]

    # On récupère plus de candidats avant de les reclasser sur le score final
    n_query = max(100, n_recommendations + 1)
    distances, indices = knn_model.kneighbors(
        knn_feature_matrix[product_index].reshape(1, -1),
        n_neighbors=n_query
    )
    indices = indices.flatten()
    distances = distances.flatten()

    # Enlever le produit lui-même
    mask = indices != product_index
    indices = indices[mask]
    distances = distances[mask]

    recommendations = df.iloc[indices].copy()

    recommendations["content_similarity"] = 1 - distances

    # Similarité de prix relative au prix du produit cible
    # (et non à l'étendue globale du catalogue)
    target_price = target["price_usd"]
    recommendations["price_similarity"] = 1 - (
        abs(recommendations["price_usd"] - target_price) / max(target_price, 1)
    )
    recommendations["price_similarity"] = recommendations["price_similarity"].clip(lower=0)

    recommendations["final_score"] = (
        0.75 * recommendations["content_similarity"]
        + 0.25 * recommendations["price_similarity"]
    )

    recommendations = recommendations.sort_values("final_score", ascending=False).head(n_recommendations)

    return recommendations[[
        "product_id", "category", "subcategory", "material", "color",
        "season", "style", "size", "price_usd", "rating", "return_rate",
        "content_similarity", "price_similarity", "final_score"
    ]]

recommend_hybrid("WC000001")


## 4. Préparation des features pour les modèles neuronaux

Encodage partagé par l'autoencoder et le modèle de Deep Metric Learning : `LabelEncoder` pour les variables catégorielles (entrée de couches `Embedding`), `StandardScaler` pour les variables numériques.


In [ ]:
cat_features = ['category', 'subcategory', 'material', 'color', 'season', 'style', 'size']
num_features = ['price_usd', 'discount_percent', 'rating', 'return_rate']

encoders = {}
cat_data = {}
cat_dims = {}

for col in cat_features:
    le = LabelEncoder()
    cat_data[col] = le.fit_transform(df[col])
    encoders[col] = le
    cat_dims[col] = len(le.classes_)

scaler = StandardScaler()
num_data = scaler.fit_transform(df[num_features]).astype("float32")

n_samples = len(df)
all_inputs = [cat_data[col].reshape(-1, 1) for col in cat_features] + [num_data]

idx_train, idx_val = train_test_split(np.arange(n_samples), test_size=0.15, random_state=42)

def subset(inputs, idx):
    return [arr[idx] for arr in inputs]

train_inputs = subset(all_inputs, idx_train)
val_inputs = subset(all_inputs, idx_val)
train_target = num_data[idx_train]
val_target = num_data[idx_val]

EMBED_DIM_CAT = 8
L2_REG = 1e-4
DROPOUT_RATE = 0.3


## 5. Approche 2 — Autoencoder multi-tâches

Version finale retenue après deux corrections successives :
- chaque variable catégorielle est reconstruite par sa propre tête `softmax` (et non par une simple reconstruction numérique globale), avec un poids de perte proportionnel au nombre de classes ($\lambda_c = \log(1 + n_c)$) ;
- la tête numérique (prix, remise, note, taux de retour) reçoit un poids de perte renforcé (`5.0` au lieu de `0.5`), car elle était sinon écrasée par les sept têtes catégorielles dans le gradient total.

Le bottleneck (32 dimensions) constitue l'embedding produit final.


In [ ]:
BOTTLENECK_DIM = 32

cat_inputs = []
cat_embeds = []

for col in cat_features:
    inp = layers.Input(shape=(1,), name=f"input_{col}")
    emb = layers.Embedding(
        input_dim=cat_dims[col],
        output_dim=EMBED_DIM_CAT,
        embeddings_regularizer=regularizers.l2(L2_REG),
        name=f"embed_{col}"
    )(inp)
    emb = layers.Flatten()(emb)
    cat_inputs.append(inp)
    cat_embeds.append(emb)

num_input = layers.Input(shape=(len(num_features),), name="input_numerical")

x = layers.Concatenate()(cat_embeds + [num_input])
x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)

bottleneck = layers.Dense(BOTTLENECK_DIM, activation=None, name="product_embedding")(x)

d = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(bottleneck)
d = layers.Dropout(DROPOUT_RATE)(d)
d = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(d)

outputs = []
losses = {}
loss_weights = {}
metrics = {}

for col in cat_features:
    out = layers.Dense(cat_dims[col], activation="softmax", name=f"recon_{col}")(d)
    outputs.append(out)
    losses[f"recon_{col}"] = "sparse_categorical_crossentropy"
    metrics[f"recon_{col}"] = "accuracy"
    loss_weights[f"recon_{col}"] = float(np.log1p(cat_dims[col]))

num_out = layers.Dense(len(num_features), activation=None, name="recon_numerical")(d)
outputs.append(num_out)
losses["recon_numerical"] = "mse"
loss_weights["recon_numerical"] = 5.0  # poids renforcé pour compenser les 7 têtes catégorielles

embedding_model = Model(inputs=cat_inputs + [num_input], outputs=bottleneck)
autoencoder = Model(inputs=cat_inputs + [num_input], outputs=outputs)
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

train_targets = {f"recon_{col}": cat_data[col][idx_train] for col in cat_features}
train_targets["recon_numerical"] = train_target
val_targets = {f"recon_{col}": cat_data[col][idx_val] for col in cat_features}
val_targets["recon_numerical"] = val_target

early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)

history_ae = autoencoder.fit(
    train_inputs, train_targets,
    validation_data=(val_inputs, val_targets),
    epochs=100, batch_size=256,
    callbacks=[early_stop, reduce_lr], verbose=1
)


In [ ]:
print("--- Accuracy de reconstruction par variable (val) ---")
for col in cat_features:
    acc_key = f"val_recon_{col}_accuracy"
    if acc_key in history_ae.history:
        print(f"{col}: {history_ae.history[acc_key][-1]:.3f}")
print("val_recon_numerical_loss:", history_ae.history["val_recon_numerical_loss"][-1])

product_embeddings_ae = embedding_model.predict(all_inputs, batch_size=512)
print("Embedding shape:", product_embeddings_ae.shape)


In [ ]:
def recommend_autoencoder(product_id, n_recommendations=5):
    """Recommandation par similarité cosinus dans l'espace latent de l'autoencoder."""
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]

    target_vec = product_embeddings_ae[product_index].reshape(1, -1)
    sims = cosine_similarity(target_vec, product_embeddings_ae).flatten()
    sims[product_index] = -np.inf

    top_indices = np.argsort(sims)[::-1][:n_recommendations]

    recommendations = df.iloc[top_indices][[
        'product_id', 'category', 'subcategory', 'material', 'color',
        'season', 'style', 'size', 'price_usd', 'rating', 'return_rate'
    ]].copy()
    recommendations['similarity_score'] = sims[top_indices]

    return recommendations.reset_index(drop=True)

recommend_autoencoder("WC000001")


## 6. Approche 3 — Neural Embedding (Deep Metric Learning, Triplet Loss)

Version finale retenue après une première itération jugée insuffisante :

| Configuration | Marge $\alpha$ | Négatifs difficiles |
|---|---|---|
| Version initiale | 0.3 | 70% |
| **Version finale** | **0.1** | **100%** |

Un **négatif difficile** est un produit de même `category` mais de `subcategory` différente. Une marge plus resserrée associée à des négatifs systématiquement difficiles force le modèle à discriminer plus finement les sous-catégories, plutôt que de se contenter de séparer les grandes catégories.

Les trois branches du modèle (anchor, positive, negative) partagent la même tour d'embedding (`shared tower`).


In [ ]:
rng = np.random.default_rng(42)

category_arr = df['category'].values
subcategory_arr = df['subcategory'].values

group_key = pd.Series(list(zip(category_arr, subcategory_arr)))
group_to_indices = group_key.groupby(group_key).apply(lambda s: s.index.to_numpy())

category_series = pd.Series(category_arr)
category_to_indices = category_series.groupby(category_series).apply(lambda s: s.index.to_numpy())

def build_triplets_hard(n_triplets):
    """Anchor aléatoire, positive = même (category, subcategory), negative = même
    category mais subcategory différente (négatif difficile, ratio 100%)."""
    anchors = rng.integers(0, n_samples, size=n_triplets)
    positives = np.empty(n_triplets, dtype=int)
    negatives = np.empty(n_triplets, dtype=int)
    skipped_self_positive = 0

    for i, a in enumerate(anchors):
        key = (category_arr[a], subcategory_arr[a])
        same_group = group_to_indices[key]

        if len(same_group) > 1:
            p = rng.choice(same_group)
            tries = 0
            while p == a and tries < 10:
                p = rng.choice(same_group)
                tries += 1
        else:
            p = a
            skipped_self_positive += 1
        positives[i] = p

        same_cat = category_to_indices[category_arr[a]]
        n = rng.choice(same_cat)
        tries = 0
        while subcategory_arr[n] == subcategory_arr[a] and tries < 30:
            n = rng.choice(same_cat)
            tries += 1
        negatives[i] = n

    print(f"Triplets anchor==positive (groupe trop petit) : {skipped_self_positive}/{n_triplets}")
    return anchors, positives, negatives

N_TRIPLETS = 200_000
anchor_idx, pos_idx, neg_idx = build_triplets_hard(N_TRIPLETS)

subcat_coverage = pd.Series(subcategory_arr[anchor_idx]).value_counts()
print("Couverture triplets par subcategory (min/max) :", subcat_coverage.min(), subcat_coverage.max())


In [ ]:
PRODUCT_EMBED_DIM = 32
MARGIN = 0.1

def build_embedding_tower():
    tower_cat_inputs = [layers.Input(shape=(1,), name=f"input_{col}") for col in cat_features]
    tower_cat_embeds = []
    for inp, col in zip(tower_cat_inputs, cat_features):
        emb = layers.Embedding(
            input_dim=cat_dims[col],
            output_dim=EMBED_DIM_CAT,
            embeddings_regularizer=regularizers.l2(L2_REG),
            name=f"embed_{col}"
        )(inp)
        tower_cat_embeds.append(layers.Flatten()(emb))

    tower_num_input = layers.Input(shape=(len(num_features),), name="input_numerical")

    x = layers.Concatenate()(tower_cat_embeds + [tower_num_input])
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)

    embedding = layers.Dense(PRODUCT_EMBED_DIM, activation=None, name="product_embedding")(x)
    embedding = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1), name="l2_normalize"
    )(embedding)

    return Model(inputs=tower_cat_inputs + [tower_num_input], outputs=embedding, name="embedding_tower")

def make_input_set(prefix):
    return [layers.Input(shape=(1,), name=f"{prefix}_{col}") for col in cat_features] + \
           [layers.Input(shape=(len(num_features),), name=f"{prefix}_numerical")]

embedding_tower = build_embedding_tower()

anchor_inputs = make_input_set("anchor")
positive_inputs = make_input_set("positive")
negative_inputs = make_input_set("negative")

anchor_emb = embedding_tower(anchor_inputs)
positive_emb = embedding_tower(positive_inputs)
negative_emb = embedding_tower(negative_inputs)

merged = layers.Concatenate(axis=1)([anchor_emb, positive_emb, negative_emb])
triplet_model = Model(inputs=anchor_inputs + positive_inputs + negative_inputs, outputs=merged)

def triplet_loss(y_true, y_pred, embed_dim=PRODUCT_EMBED_DIM, margin=MARGIN):
    a = y_pred[:, :embed_dim]
    p = y_pred[:, embed_dim:2*embed_dim]
    n = y_pred[:, 2*embed_dim:]
    pos_dist = tf.reduce_sum(tf.square(a - p), axis=1)
    neg_dist = tf.reduce_sum(tf.square(a - n), axis=1)
    return tf.reduce_mean(tf.maximum(pos_dist - neg_dist + margin, 0.0))

triplet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=triplet_loss)


In [ ]:
def gather_inputs(indices):
    return [cat_data[col][indices].reshape(-1, 1) for col in cat_features] + [num_data[indices]]

train_data = gather_inputs(anchor_idx) + gather_inputs(pos_idx) + gather_inputs(neg_idx)
dummy_y = np.zeros((N_TRIPLETS, 1))  # y_true inutilisé : la loss dépend uniquement de y_pred

early_stop = callbacks.EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5, patience=2, min_lr=1e-5)

history_nn = triplet_model.fit(
    train_data, dummy_y,
    epochs=40, batch_size=512,
    callbacks=[early_stop, reduce_lr], verbose=1
)

product_embeddings_nn = embedding_tower.predict(all_inputs, batch_size=1024)
print("Embedding shape:", product_embeddings_nn.shape)


In [ ]:
def recommend_neural_embedding(product_id, n_recommendations=5):
    """Recommandation par similarité cosinus dans l'espace d'embedding appris par Triplet Loss."""
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]

    target_vec = product_embeddings_nn[product_index].reshape(1, -1)
    sims = cosine_similarity(target_vec, product_embeddings_nn).flatten()
    sims[product_index] = -np.inf

    top_indices = np.argsort(sims)[::-1][:n_recommendations]

    recommendations = df.iloc[top_indices][[
        'product_id', 'category', 'subcategory', 'material', 'color',
        'season', 'style', 'size', 'price_usd', 'rating', 'return_rate'
    ]].copy()
    recommendations['similarity_score'] = sims[top_indices]

    return recommendations.reset_index(drop=True)

recommend_neural_embedding("WC000001")


## 7. Diagnostic de la qualité des embeddings

Une similarité cosinus moyenne proche de 1.0 avec un écart-type proche de 0 signale un effondrement de l'espace latent (l'autoencoder n'a pas discriminé les produits) : c'est ce qui a été observé dans la toute première version de l'autoencoder, avant correction.


In [ ]:
def check_embedding_health(embeddings, name):
    pairwise_sample = cosine_similarity(embeddings[:500])
    off_diag = pairwise_sample[~np.eye(500, dtype=bool)]
    print(f"[{name}] std par dimension (moyenne) : {embeddings.std(axis=0).mean():.4f}")
    print(f"[{name}] similarité cosinus moyenne (paires aléatoires) : {off_diag.mean():.4f}")
    print(f"[{name}] similarité cosinus std : {off_diag.std():.4f}")

check_embedding_health(product_embeddings_ae, "Autoencoder")
check_embedding_health(product_embeddings_nn, "Neural Embedding")


## 8. Évaluation comparative

Les trois modèles sont comparés sur un échantillon de 200 produits tirés aléatoirement, avec $K=5$ recommandations par produit, selon trois métriques :
- **Category Match@5** : proportion de recommandations partageant la catégorie du produit cible
- **Subcategory Match@5** : proportion partageant la sous-catégorie
- **Écart de prix relatif moyen**


In [ ]:
def evaluate_model(recommend_fn, n_products=200, k=5):
    sample_ids = df['product_id'].sample(n_products, random_state=42).values
    cat_match, subcat_match, price_ratios = [], [], []

    for pid in sample_ids:
        target_row = df[df['product_id'] == pid].iloc[0]
        recs = recommend_fn(pid, k)
        if recs is None or len(recs) == 0:
            continue
        cat_match.append((recs['category'] == target_row['category']).mean())
        subcat_match.append((recs['subcategory'] == target_row['subcategory']).mean())
        price_ratios.append((abs(recs['price_usd'] - target_row['price_usd']) / max(target_row['price_usd'], 1)).mean())

    return {
        "category_match@5": np.mean(cat_match),
        "subcategory_match@5": np.mean(subcat_match),
        "price_gap": np.mean(price_ratios),
    }

results = {
    "KNN hybride": evaluate_model(lambda pid, k: recommend_hybrid(pid, k)),
    "Autoencoder": evaluate_model(lambda pid, k: recommend_autoencoder(pid, k)),
    "Neural Embedding": evaluate_model(lambda pid, k: recommend_neural_embedding(pid, k)),
}

results_df = pd.DataFrame(results).T
results_df.columns = ["Category Match@5", "Subcategory Match@5", "Price Gap"]
results_df.round(3)


## 9. Comparaison qualitative

Exemple sur un produit cible unique, pour visualiser concrètement les différences de comportement entre les trois modèles.


In [ ]:
def compare_all(product_id, n_recommendations=5):
    print(f"=== Cible : {product_id} ===\n")

    knn_res = recommend_hybrid(product_id, n_recommendations)
    ae_res = recommend_autoencoder(product_id, n_recommendations)
    nn_res = recommend_neural_embedding(product_id, n_recommendations)

    print("--- KNN hybride ---")
    print(knn_res[['product_id', 'subcategory', 'color', 'price_usd']])
    print("\n--- Autoencoder ---")
    print(ae_res[['product_id', 'subcategory', 'color', 'price_usd']])
    print("\n--- Neural Embedding (Triplet Loss) ---")
    print(nn_res[['product_id', 'subcategory', 'color', 'price_usd']])

    knn_set, ae_set, nn_set = set(knn_res['product_id']), set(ae_res['product_id']), set(nn_res['product_id'])
    print(f"\nRecouvrement KNN ∩ Autoencoder : {knn_set & ae_set}")
    print(f"Recouvrement KNN ∩ Neural Embedding : {knn_set & nn_set}")
    print(f"Recouvrement Autoencoder ∩ Neural Embedding : {ae_set & nn_set}")

    return knn_res, ae_res, nn_res

compare_all("WC000001")


## 10. Conclusion

Le **KNN hybride** obtient la meilleure cohérence de catégorie et le plus faible écart de prix (attendu, puisque le prix est intégré explicitement dans son score). L'**autoencoder**, une fois corrigé pour reconstruire séparément chaque variable catégorielle, atteint une forte cohérence de sous-catégorie mais maîtrise mal le prix, resté un signal marginal dans son espace latent. Le **Neural Embedding**, entraîné avec des négatifs 100% difficiles et une marge resserrée, obtient la meilleure cohérence de sous-catégorie des trois, au prix d'un écart de prix légèrement supérieur à celui du KNN — cohérent avec une fonction de perte qui n'intègre pas le prix.

Aucun modèle ne domine sur tous les critères : le choix dépend de la notion de similarité que l'on souhaite privilégier.

**Limite principale** : les produits évalués appartiennent au même catalogue que celui utilisé pour l'entraînement (construction des triplets). Une évaluation sur un ensemble de test totalement disjoint serait nécessaire pour exclure tout effet de mémorisation.
